In [ ]:
import numpy
import pandas


# Constants
__TRAINING_DATAFRAME__ = pandas.read_csv("/kaggle/input/playground-series-s4e10/train.csv")
__TESTING_DATAFRAME__  = pandas.read_csv("/kaggle/input/playground-series-s4e10/test.csv")

__SEED__ = 123

In [ ]:
__TRAINING_DATAFRAME__.head()

In [ ]:
__TRAINING_DATAFRAME__.info()

In [ ]:
training_dataframe = __TRAINING_DATAFRAME__
training_dataframe = training_dataframe.drop("id", axis = "columns")

categorical_data_columns = [
    "person_home_ownership",
    "loan_intent",
    "loan_grade",
    "cb_person_default_on_file"
]

for categorical_column in categorical_data_columns:
    encoded_column = pandas.get_dummies(training_dataframe[categorical_column], prefix = categorical_column, prefix_sep = "__", dtype = int)
    training_dataframe = training_dataframe.drop(categorical_column, axis = "columns")
    training_dataframe = training_dataframe.join(encoded_column)

training_dataframe.info()

In [ ]:
# Histogrammable plots
def generate_loan_approval_plot(training_dataframe):
    import plotly
    import plotly.subplots

    histogrammable_columns = [
            "person_age", 
            "person_income", 
            "person_emp_length", 
            "loan_amnt", 
            "loan_int_rate",
            "loan_percent_income",
            "cb_person_cred_hist_length",
        ]
    categorical_columns = [[__ for __ in training_dataframe.columns if _ in __] for _ in categorical_data_columns]

    plot_count = 0

    figure = plotly.subplots.make_subplots(rows = 4, cols = 3)

    for histogrammable_column in histogrammable_columns:
        col = int(plot_count / 4) + 1  
        row = (plot_count % 4) + 1 
        plot_count += 1

        figure.add_trace(
            plotly.graph_objects.Histogram(
                x=training_dataframe[histogrammable_column], 
                autobinx=True,
                marker = {
                    "color": "#A7C7E7"
                }
            ), 
            row=row, 
            col=col,
        )
        figure.update_xaxes(title_text=f"{histogrammable_column}", row=row, col=col)
        figure.update_yaxes(title_text="Frequency", row=row, col=col)

    for categorical_column in categorical_columns:
        col = int(plot_count / 4) + 1  
        row = (plot_count % 4) + 1    
        plot_count += 1

        figure.add_trace(
            plotly.graph_objects.Bar(
                y = [len(training_dataframe[d].loc[training_dataframe[d] == 1]) for d in categorical_column],
                x = [s.split('__', 1)[1] for s in categorical_column],
                marker = {
                    "color": "#A7C7E7"
                }
            ),
            row=row,
            col=col
        )
        figure.update_yaxes(title_text="Count", row=row, col=col)

    col = int(plot_count / 4) + 1  
    row = (plot_count % 4) + 1    
    figure.add_trace(
        plotly.graph_objects.Bar(
            y = [
                len(training_dataframe["loan_status"].loc[training_dataframe["loan_status"] == 0]),
                len(training_dataframe["loan_status"].loc[training_dataframe["loan_status"] == 1]),
                ],
            x = ["0", "1"],
            marker = {
                "color": "#A7C7E7"
            }
        ),
        row=row,
        col=col
    )
    figure.update_yaxes(title_text="Count", row=row, col=col)
    figure.update_layout(height = 1200, width = 1200, title_text = "Data Distribution", showlegend = False)
    
    return figure
generate_loan_approval_plot(training_dataframe).show()

# 

In [ ]:
import sklearn.preprocessing

normalizer = sklearn.preprocessing.StandardScaler()
normalizer_columns = [
            "person_age", 
            "person_income", 
            "person_emp_length", 
            "loan_amnt", 
            "loan_int_rate",
            "loan_percent_income",
            "cb_person_cred_hist_length",
        ]
training_dataframe[normalizer_columns] = pandas.DataFrame(
    normalizer.fit_transform(training_dataframe[normalizer_columns])
)

generate_loan_approval_plot(training_dataframe).show()

In [ ]:
print(training_dataframe.shape)

In [ ]:
import sklearn.model_selection

training_dataframe, validation_dataframe = sklearn.model_selection.train_test_split(
    training_dataframe, test_size = 0.2, random_state = __SEED__
)

In [ ]:
testing_dataframe = __TESTING_DATAFRAME__
testing_id = testing_dataframe["id"]
testing_dataframe = testing_dataframe.drop("id", axis = "columns")

categorical_data_columns = [
    "person_home_ownership",
    "loan_intent",
    "loan_grade",
    "cb_person_default_on_file"
]

for categorical_column in categorical_data_columns:
    encoded_column = pandas.get_dummies(testing_dataframe[categorical_column], prefix = categorical_column, prefix_sep = "__", dtype = int)
    testing_dataframe = testing_dataframe.drop(categorical_column, axis = "columns")
    testing_dataframe = testing_dataframe.join(encoded_column)

testing_dataframe[normalizer_columns] = pandas.DataFrame(
    normalizer.transform(testing_dataframe[normalizer_columns])
)


# Training and Inference

In [ ]:
__ACTIVE_MODELS__ = [
#     "logistic_regression",
#     "svm",
#     "knn",
#     "random_forest",
#     "ada_boost",
#     "gradient_boosting",
    "catboost"
]

__ACTIVE_MODEL_PARAMETERS__ = {
    "logistic_regression" : {
        "max_iter": 1000,
        "save": True,
        "validate": True
    },
    
    "svm": {
        
    },
    
    "knn": {
        "n_neighbors": 5,
        "weights": 'uniform', # 'uniform', 'distance'
        "algorithm": 'auto', # 'ball_tree', 'kd_tree', 'brute', 'auto'
        "leaf_size": 30,
        "p": 2,
        "metric": 'minkowski',
        "n_jobs": 2,    
        "save": True
    },
    
    "random_forest": {
        "n_estimators": 300,
        "criterion": "entropy", # {“gini”, “entropy”, “log_loss”}
        "validate": True,
        "save": True,
    },
    
    "ada_boost": {
        "n_estimators": 200,
        "learning_rate": 1.0,
        "algorithm": 'SAMME.R', #{‘SAMME’, ‘SAMME.R’},
        "random_state": __SEED__,
        "validate": True,
        "save": True
    }
}

In [ ]:
def roc_score(y, y_, plot = False):
    import sklearn.metrics 
    import plotly
    
    roc_score = sklearn.metrics.roc_auc_score(y, y_)
    print(f"ROC Score: {roc_score}")
    
    if plot:
        fpr, tpr, thresholds = sklearn.metrics.roc_curve(y, y_)

        fig = plotly.graph_objects.Figure()

        fig.add_trace(plotly.graph_objects.Scatter(x=fpr, y=tpr, mode='lines', name=f'ROC curve (AUC = {roc_score:.4f})',
                                 line=dict(color='blue', width=2)))
        fig.add_trace(plotly.graph_objects.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random classifier',
                                 line=dict(color='gray', dash='dash')))

        fig.update_layout(title='Receiver Operating Characteristic (ROC) Curve',
                          xaxis_title='False Positive Rate',
                          yaxis_title='True Positive Rate',
                          xaxis=dict(range=[0, 1]),
                          yaxis=dict(range=[0, 1]),
                          showlegend=True,
                          template='plotly_white')

        fig.show()

## Logistic Regression

In [ ]:
if "logistic_regression" in __ACTIVE_MODELS__:
    import sklearn.linear_model
    
    __MODEL_PARAMETERS = __ACTIVE_MODEL_PARAMETERS__["logistic_regression"]      
    __max_iter = __MODEL_PARAMETERS["max_iter"]
    __validate = __MODEL_PARAMETERS["validate"]
    

    classifier = sklearn.linear_model.LogisticRegressionCV(max_iter = __max_iter)
    y = training_dataframe["loan_status"]
    X = training_dataframe.drop("loan_status", axis = "columns")

    classifier.fit(X, y)
    
    if __validate:
        y = validation_dataframe["loan_status"]
        x = validation_dataframe.drop("loan_status", axis = "columns")
        
        y_ = pandas.DataFrame(classifier.predict_proba(x)[:, 1])
        
        roc_score(y, y_, True)
        
    out = pandas.DataFrame(classifier.predict_proba(testing_dataframe))
    res = pandas.concat([pandas.DataFrame(testing_id), out], axis = 1)
    res.columns = ["id", "loan_status_1", "loan_status"]
    res[["id", "loan_status"]].to_csv("submission.csv", index=False)

## SVM

In [ ]:
if "svm" in __ACTIVE_MODELS__:
    import sklearn.svm

    classifier = sklearn.svm.SVC(probability = True)
    y = training_dataframe["loan_status"]
    X = training_dataframe.drop("loan_status", axis = "columns")

    classifier.fit(X, y)

    out = pandas.DataFrame(classifier.predict_proba(testing_dataframe))
    res = pandas.concat([pandas.DataFrame(testing_id), out], axis = 1)
    res.columns = ["id", "loan_status_1", "loan_status"]
    res[["id", "loan_status"]].to_csv("submission.csv", index=False)

## KNeighbors

In [ ]:
if "knn" in __ACTIVE_MODELS__:
    import sklearn.neighbors
    
    MODEL_PARAMETERS = __ACTIVE_MODEL_PARAMETERS__["knn"]        
    __n_neighbors = MODEL_PARAMETERS["n_neighbors"]
    __weights = MODEL_PARAMETERS["weights"]
    __algorithm = MODEL_PARAMETERS["algorithm"]
    __leaf_size = MODEL_PARAMETERS["leaf_size"]
    __p = MODEL_PARAMETERS["p"]
    __metric = MODEL_PARAMETERS["metric"]
    __n_jobs = MODEL_PARAMETERS["n_jobs"]
    
    __save = MODEL_PARAMETERS["save"]
    
    classifier = sklearn.neighbors.KNeighborsClassifier(
        __n_neighbors,
        weights = __weights,
        algorithm = __algorithm,
        leaf_size = __leaf_size,
        p = __p,
        metric = __metric,
        n_jobs = __n_jobs
    )
    
    y = training_dataframe["loan_status"]
    X = training_dataframe.drop("loan_status", axis = "columns")

    classifier.fit(X, y)

    out = pandas.DataFrame(classifier.predict_proba(testing_dataframe))
    res = pandas.concat([pandas.DataFrame(testing_id), out], axis = 1)
    
    if __save:
        res.columns = ["id", "loan_status_1", "loan_status"]
        res[["id", "loan_status"]].to_csv("submission.csv", index=False)
    

## Random Forest

In [ ]:
if "random_forest" in __ACTIVE_MODELS__:
    import sklearn.ensemble 
    
    __MODEL_PARAMETERS = __ACTIVE_MODEL_PARAMETERS__["random_forest"]
    __n_estimators = __MODEL_PARAMETERS["n_estimators"]
    __criterion = __MODEL_PARAMETERS["criterion"]
    __validate = __MODEL_PARAMETERS["validate"]
    __save = __MODEL_PARAMETERS["save"]
    
    classifier = sklearn.ensemble.RandomForestClassifier(
        n_estimators = __n_estimators, criterion = __criterion
    )
    
    y = training_dataframe["loan_status"]
    X = training_dataframe.drop("loan_status", axis = "columns")
    
    classifier.fit(X, y)
    
    if __validate:
        y = validation_dataframe["loan_status"]
        x = validation_dataframe.drop("loan_status", axis = "columns")
        
        y_ = pandas.DataFrame(classifier.predict_proba(x)[:, 1])
        
        roc_score(y, y_, True)
    
    if __save:
        out = pandas.DataFrame(classifier.predict_proba(testing_dataframe))
        res = pandas.concat([pandas.DataFrame(testing_id), out], axis = 1)
        res.columns = ["id", "loan_status_1", "loan_status"]
        res[["id", "loan_status"]].to_csv("submission.csv", index=False)

In [ ]:
if "ada_boost" in __ACTIVE_MODELS__:
    import sklearn.ensemble 
    
    __MODEL_PARAMETERS = __ACTIVE_MODEL_PARAMETERS__["ada_boost"]
    __n_estimators = __MODEL_PARAMETERS["n_estimators"]
    __learning_rate = __MODEL_PARAMETERS["learning_rate"]
    __algorithm = __MODEL_PARAMETERS["algorithm"]
    __random_state = __MODEL_PARAMETERS["random_state"]
    __validate = __MODEL_PARAMETERS["validate"]
    __save = __MODEL_PARAMETERS["save"]
    
    classifier = sklearn.ensemble.AdaBoostClassifier(
        n_estimators = __n_estimators, learning_rate = __learning_rate, algorithm = __algorithm, random_state = __random_state
    )
    
    y = training_dataframe["loan_status"]
    X = training_dataframe.drop("loan_status", axis = "columns")
    
    classifier.fit(X, y)
    
    if __validate:
        y = validation_dataframe["loan_status"]
        x = validation_dataframe.drop("loan_status", axis = "columns")
        
        y_ = pandas.DataFrame(classifier.predict_proba(x)[:, 1])
        
        roc_score(y, y_, True)
    
    if __save:
        out = pandas.DataFrame(classifier.predict_proba(testing_dataframe))
        res = pandas.concat([pandas.DataFrame(testing_id), out], axis = 1)
        res.columns = ["id", "loan_status_1", "loan_status"]
        res[["id", "loan_status"]].to_csv("submission.csv", index=False)
    

In [ ]:
if "gradient_boosting" in __ACTIVE_MODELS__:
    import sklearn.ensemble
    
    classifier = sklearn.ensemble.GradientBoostingClassifier(loss='exponential', learning_rate=0.1, n_estimators=100, subsample=1.0, criterion='friedman_mse', min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_depth=3, min_impurity_decrease=0.0, init=None, random_state=None, max_features=None, verbose=0, max_leaf_nodes=None, warm_start=False, validation_fraction=0.1, n_iter_no_change=None, tol=0.0001, ccp_alpha=0.0)
    
    y = training_dataframe["loan_status"]
    X = training_dataframe.drop("loan_status", axis = "columns")
    
    classifier.fit(X, y)
    
    if True:
        y = validation_dataframe["loan_status"]
        x = validation_dataframe.drop("loan_status", axis = "columns")
        
        y_ = pandas.DataFrame(classifier.predict_proba(x)[:, 1])
        
        roc_score(y, y_, True)
    
    if False:
        out = pandas.DataFrame(classifier.predict_proba(testing_dataframe))
        res = pandas.concat([pandas.DataFrame(testing_id), out], axis = 1)
        res.columns = ["id", "loan_status_1", "loan_status"]
        res[["id", "loan_status"]].to_csv("submission.csv", index=False)

In [ ]:
if "gradient_boosting" in __ACTIVE_MODELS__:
    import sklearn.ensemble
    import sklearn.model_selection
    
    classifier = sklearn.ensemble.GradientBoostingClassifier(loss='exponential', learning_rate=0.1, n_estimators=100, subsample=1.0, criterion='friedman_mse', min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_depth=3, min_impurity_decrease=0.0, init=None, random_state=None, max_features=None, verbose=0, max_leaf_nodes=None, warm_start=False, validation_fraction=0.1, n_iter_no_change=None, tol=0.0001, ccp_alpha=0.0)
    
    y = training_dataframe["loan_status"]
    X = training_dataframe.drop("loan_status", axis = "columns")
    parameters = {
        'loss': ('log_loss', ),
        'learning_rate': [0.1,],
        'n_estimators': [200],
        'subsample': [1.0],
        'criterion': ('friedman_mse', ),
        #'min_samples_split': [0.1, 0.5, 1.0],
        #'min_samples_leaf': [0.1, 0.5, 1.0],
        #'min_weight_fraction_leaf': [0.0, 0.25, 0.5],
        #'max_depth': [1, 3, 5, 10, 50],
        #'min_impurity_decrease': [0.0, 0.5, 1.0, 2.5, 5.0, 10.0]
    }
    
    classifier_ = sklearn.model_selection.GridSearchCV(
        classifier,
        parameters
    )
    
    
    classifier_.fit(X, y)
    
    # Get the best parameters and best score
    best_params = classifier_.best_params_
    best_score = classifier_.best_score_

    print("Best Parameters:", best_params)
    print("Best Score:", best_score)
    
    if True:
        y = validation_dataframe["loan_status"]
        x = validation_dataframe.drop("loan_status", axis = "columns")
        
        y_ = pandas.DataFrame(classifier_.predict_proba(x)[:, 1])
        
        roc_score(y, y_, True)
    
    if True:
        out = pandas.DataFrame(classifier_.predict_proba(testing_dataframe))
        res = pandas.concat([pandas.DataFrame(testing_id), out], axis = 1)
        res.columns = ["id", "loan_status_1", "loan_status"]
        res[["id", "loan_status"]].to_csv("submission.csv", index=False)

In [ ]:
if "catboost" in __ACTIVE_MODELS__:
    import catboost

    y = training_dataframe["loan_status"]
    X = training_dataframe.drop("loan_status", axis = "columns")
    #classifier = catboost.CatBoostClassifier(iterations = 1000, depth = 8, learning_rate = 0.1, loss_function = 'Logloss', verbose = 100)
    classifier = catboost.CatBoostClassifier(iterations = 2500, depth = 8, learning_rate = 0.1, loss_function = 'Logloss', verbose = 100)
    
    classifier.fit(X, y)
    
    if True:
        y = validation_dataframe["loan_status"]
        x = validation_dataframe.drop("loan_status", axis = "columns")
        
        y_ = pandas.DataFrame(classifier.predict_proba(x)[:, 1])
        
        roc_score(y, y_, True)
    
    y = pandas.concat([training_dataframe["loan_status"], validation_dataframe["loan_status"]], axis=0)
    X = pandas.concat([training_dataframe.drop("loan_status", axis = "columns"), validation_dataframe.drop("loan_status", axis = "columns")], axis=0)
    
    classifier.fit(X, y)
    
    if True:
        out = pandas.DataFrame(classifier.predict_proba(testing_dataframe))
        res = pandas.concat([pandas.DataFrame(testing_id), out], axis = 1)
        res.columns = ["id", "loan_status_1", "loan_status"]
        res[["id", "loan_status"]].to_csv("submission.csv", index=False)